In [ ]:
import os

import numpy as np
import pandas as pd
import torch
from scipy.stats import sem


def mae_and_sem(preds: np.ndarray, trues: np.ndarray):
    abs_error = np.abs(preds - trues)
    mae = np.mean(abs_error)
    mae_sem = sem(abs_error)
    return mae, mae_sem


def load_outputs_labels_ppg2bpnet(logdir: str):
    results = torch.load(os.path.join(logdir, "best_model.pth"))
    pred_bps = results["test_pred"]
    gt_bps = results["test_label"]
    
    return pred_bps, gt_bps


def load_outputs_labels_ttc(logdir: str):
    gt_bps = np.load(os.path.join(logdir, "gt.npy"))
    pred_bps = np.load(os.path.join(logdir, "pred.npy"))
    is_calibrated = np.load(os.path.join(logdir, "is_labeled.npy"))

    return pred_bps[~is_calibrated], gt_bps[~is_calibrated]


In [ ]:
test_index_path = "../data/index/test.parquet"
ppg2bpnet_logdir = "../calib_based/outputs/calib_based/ppg2bpnet"
ttc_logdir = "../calib_based/outputs/calib_based/ttc"

In [ ]:
# 1) PPG2BPNet
pred_bps, gt_bps = load_outputs_labels_ppg2bpnet(ppg2bpnet_logdir)
print("PPG2BPNet evaluation (MAE)")
for i, name in enumerate(["SBP", "DBP"]):
    mae, mae_sem = mae_and_sem(pred_bps[:, i], gt_bps[:, i])
    print(f"\t{name}: {mae:.2f} ± {1.96 * mae_sem:.2f}")

# 2) TTC
pred_bps, gt_bps = load_outputs_labels_ttc(ttc_logdir)
print("TTC evaluation (MAE)")
for i, name in enumerate(["SBP", "DBP"]):
    mae, mae_sem = mae_and_sem(pred_bps[:, i], gt_bps[:, i])
    print(f"\t{name}: {mae:.2f} ± {1.96 * mae_sem:.2f}")
